# Think Vector Analysis
Difference-in-means representation: `v = mean_hidden(Explicit) − mean_hidden(Implicit)`

Captures what the model encodes differently about mental-state language vs. action language (Marks & Tegmark 2023).

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ROOT    = Path().resolve().parent
TV_DIR  = ROOT / 'results' / 'think_vectors'
sys.path.insert(0, str(ROOT / 'src'))

CONDITIONS = ['flat', 'chunked', 'balanced']
LABELS  = {'flat': 'Sliding-Window', 'chunked': 'EOS-Chunked', 'balanced': 'DENSITY-Balanced'}
COLORS  = {'flat': '#E64B35', 'chunked': '#4DBBD5', 'balanced': '#00A087'}

def load_summary(cond):
    p = TV_DIR / cond / 'think_vector_summary.json'
    if not p.exists():
        print(f'[{cond}] not found — run: sbatch jobs/job_extract_think_vectors.sh {cond}')
        return None
    data = json.loads(p.read_text())
    return sorted(data.get('checkpoints', []), key=lambda r: r['step'])

summaries = {c: load_summary(c) for c in CONDITIONS}
for c, v in summaries.items():
    if v: print(f'{c}: {len(v)} checkpoints extracted')

## What is a think vector?

For each checkpoint, we pass each FBT stimulus through the model and record the **last-token hidden state** at every transformer layer.

The think vector at layer L is:
```
v_L = mean(hidden states of Explicit items) − mean(hidden states of Implicit items)
```
If the model encodes mental-state language differently from action language, this vector will have a stable direction. We track **cosine similarity to the final checkpoint** as a proxy for when the representation converges.

## Vector norm over training (all conditions)

In [ ]:
FOCUS_LAYER = 6  # mid-network layer; adjust if needed

fig, ax = plt.subplots(figsize=(10, 4))
for cond in CONDITIONS:
    recs = summaries.get(cond)
    if not recs: continue
    xs    = [r['words_seen'] / 1e6 for r in recs if r.get('words_seen')]
    norms = [r['norms'][FOCUS_LAYER] for r in recs if r.get('words_seen')]
    ax.plot(xs, norms, color=COLORS[cond], lw=2, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel(f'‖think vector‖  (L{FOCUS_LAYER})')
ax.set_title('Think Vector Magnitude Over Training', fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Cosine similarity to final checkpoint (directional convergence)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0,   color='#ccc', lw=1, ls='--')
ax.axhline(0.5, color='#ccc', lw=0.8, ls=':')
ax.axhline(1.0, color='#ccc', lw=0.8, ls=':')

for cond in CONDITIONS:
    recs = summaries.get(cond)
    if not recs: continue
    cos_recs = [r for r in recs if r.get('words_seen') and r.get('cosine_sim_to_final')]
    if not cos_recs: continue
    xs     = [r['words_seen'] / 1e6 for r in cos_recs]
    cosims = [r['cosine_sim_to_final'][FOCUS_LAYER] for r in cos_recs]
    ax.plot(xs, cosims, color=COLORS[cond], lw=2, marker='o', markersize=2.5, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel(f'CosSim to final  (L{FOCUS_LAYER})')
ax.set_ylim(-0.15, 1.1)
ax.set_title('Think Vector Direction Convergence Over Training', fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## All-layer heatmap — which layer leads?

In [ ]:
for cond in CONDITIONS:
    recs = summaries.get(cond)
    if not recs: continue
    cos_recs = [r for r in recs if r.get('words_seen') and r.get('cosine_sim_to_final')]
    if not cos_recs: continue

    xs   = np.array([r['words_seen'] / 1e6 for r in cos_recs])
    mat  = np.array([r['cosine_sim_to_final'] for r in cos_recs]).T  # (n_layers, n_ckpts)

    if mat.shape[1] > 200:
        idx = np.linspace(0, mat.shape[1]-1, 200, dtype=int)
        mat, xs = mat[:, idx], xs[idx]

    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(mat, aspect='auto', origin='lower',
                   extent=[xs[0], xs[-1], -0.5, mat.shape[0]-0.5],
                   cmap='RdYlGn', vmin=-0.2, vmax=1.0)
    ax.axhline(FOCUS_LAYER, color='white', lw=1.5, ls='--', alpha=0.7)
    plt.colorbar(im, ax=ax, fraction=0.02).set_label('CosSim to final')
    ax.set_xlabel('Words Seen (M)')
    ax.set_ylabel('Layer')
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels([f'L{i}' for i in range(mat.shape[0])], fontsize=7)
    ax.set_title(f'Think Vector Stability by Layer — {LABELS[cond]}', fontweight='bold')
    for sp in ax.spines.values(): sp.set_visible(False)
    plt.tight_layout()
    plt.show()

## Overlay: think vector convergence vs FBT behaviour

In [ ]:
for cond in CONDITIONS:
    recs = summaries.get(cond)
    if not recs: continue
    cos_recs = [r for r in recs if r.get('words_seen') and r.get('cosine_sim_to_final')]
    if not cos_recs: continue

    fbt_path = ROOT / 'results' / cond / 'checkpoint_evals.json'
    if not fbt_path.exists(): continue
    fbt_data  = json.loads(fbt_path.read_text())
    fbt_ckpts = sorted([c for c in fbt_data.get('checkpoints',[]) if c.get('words_seen')],
                       key=lambda c: c['words_seen'])

    xs_cos  = [r['words_seen'] / 1e6 for r in cos_recs]
    cosims  = [r['cosine_sim_to_final'][FOCUS_LAYER] for r in cos_recs]
    xs_fbt  = [c['words_seen'] / 1e6 for c in fbt_ckpts]
    fb      = [c['fbt_false_belief']  for c in fbt_ckpts]
    tb      = [c['fbt_true_belief']   for c in fbt_ckpts]

    fig, ax1 = plt.subplots(figsize=(11, 4.5))
    ax2 = ax1.twinx()

    ax1.plot(xs_cos, cosims, color='#47A9A9', lw=2, marker='o', markersize=2.5,
             label=f'Think vector CosSim (L{FOCUS_LAYER})')
    ax1.set_ylabel(f'CosSim to final  (L{FOCUS_LAYER})', color='#47A9A9')
    ax1.tick_params(axis='y', labelcolor='#47A9A9')
    ax1.set_ylim(-0.15, 1.1)

    ax2.plot(xs_fbt, fb, color='#E64B35', lw=1.8, label='False Belief')
    ax2.plot(xs_fbt, tb, color='#4DBBD5', lw=1.8, label='True Belief')
    ax2.axhline(0.5, color='#aaa', lw=1, ls='--')
    ax2.set_ylabel('P(Correct)')
    ax2.set_ylim(0.1, 1.0)

    ax1.set_xlabel('Words Seen (M)')
    ax1.set_title(f'Think Vector vs FBT Behaviour — {LABELS[cond]}', fontweight='bold')
    lines1, lab1 = ax1.get_legend_handles_labels()
    lines2, lab2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, lab1+lab2, fontsize=9, loc='lower right')
    plt.tight_layout()
    plt.show()

## Emergence thresholds

In [ ]:
import pandas as pd
rows = []
for cond in CONDITIONS:
    recs = summaries.get(cond)
    if not recs: continue
    cos_recs = [r for r in recs if r.get('words_seen') and r.get('cosine_sim_to_final')]
    if not cos_recs: continue
    for thresh in [0.1, 0.5, 0.9]:
        hit = next((r['words_seen']/1e6 for r in cos_recs
                    if r['cosine_sim_to_final'][FOCUS_LAYER] >= thresh), None)
        rows.append({'Condition': LABELS[cond], f'CosSim≥{thresh}': f'{hit:.1f}M' if hit else 'N/A'})

if rows:
    df = pd.DataFrame(rows).groupby('Condition').first().reset_index()
    df